# Compare growth rates of nude admixture mice to an additive model

This file generates Figure 3b in the text and the data for Supplementary Table 1.

It requires as input the file containing the tumor growth data, a config file, and the previously estimated exponential growth rates of the admixture data as input and will save the figure to the path if specified (change file path below as needed).

The code will estimate the growth rates of the additive model for each admixture. It will then create a plot comparing the growth rates estimated from the true data to the growth rates estimated from the additive model.

## Prep

Load required packages

In [ ]:
%matplotlib widget
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import os
from sklearn.metrics import mean_squared_error
from estimator import Estimator

Close any figures from previous runs

In [ ]:
plt.close("all")

## Define data and user-input parameters

Define the data file, config file, save files, and admixture group names.

In [ ]:
data_file = "data/growth_data_scaled.csv"
admix_result_path = "results/growth/"
config = "config_subclone.json"
fig_save_file = "figures/add.svg" # Set to None to not save the figure

nude_groups = ["Grp. B2 nude (80% C1; 20% C11)", "Grp. B3 nude (50% C1; 50% C11)", "Grp. B4 nude (20% C1; 80% C11)"]

Create an estimator object from the config file

In [ ]:
es = Estimator(config)

Load the growth data

In [ ]:
growth_df = pd.read_csv(data_file)

Previously estimated growth rates for the admixture growth curves

In [ ]:
# Uncomment and run this to estimate values from your own results

# admix_results = []
# for fname in [f for f in os.listdir(admix_result_path) if ".csv" in f]:
#     if "admix_" in fname:
#         admix_results += [pd.read_csv("{}{}".format(admix_result_path, fname))]
# admix_results = pd.concat(admix_results)
# avg_admix_results = admix_results.groupby(["group", "g"])["error"].mean().reset_index()
# admix_grs = avg_admix_results.loc[avg_admix_results.groupby("group")["error"].idxmin()]["g"].tolist()
# admix_grs = np.append(0.135, admix_grs)
# admix_grs = np.append(admix_grs, 0.107)

In [ ]:
# These are our estimated results -- Comment this if you are estimating your own values in the block above
admix_grs = [0.135, 0.134, 0.134, 0.125, 0.107]

Helper function to get the ratio of C1 from the group name

In [ ]:
def get_ratio(group):
    if "A2" in group or "B2" in group: return 0.8
    if "A3" in group or "B3" in group: return 0.5
    if "A4" in group or "B4" in group: return 0.2

## Estimate the growth rates of the additive model for each admixture

Define the range of growth rates to use for the exhaustive search to find the optimal growth rate of the additive model.

In [ ]:
growth_rates = np.arange(0.1, 0.14, 0.001)

For each admixture, generate the curve that is the result of an additive model. Do this by first generating the exponential curve for each subclone as defined by that subclone's growth rate. Then generate the total tumor curve as being equal to the ratio of C1 * size of estimated exponential curve of C1 + the ratio of C11 * size of estimated exponential curve of C11.

Once this curve is generated, us an exhaustive search over the growth rates defined above to determine the optimal overall growth rate of the curve generated by the additive model. Store this information in a list.

In [ ]:
add_grs = []
for gid in range(len(nude_groups)):
    print(nude_groups[gid])
    r = get_ratio(nude_groups[gid])
    bx = growth_df[growth_df["group"] == nude_groups[gid]]
    days = bx["day"]
    c1 = np.asarray([np.exp(0.135*t) for t in days])
    c11 = np.asarray([np.exp(0.107*t) for t in days])
    c1_scaled = r*c1
    c11_scaled = (1-r)*c11
    total_scaled = c1_scaled + c11_scaled
    mses = []
    for g in growth_rates:
        generated = [np.exp(g*t) for t in days]
        mses += [mean_squared_error(total_scaled, generated)]
    # print("Additive growth rate: {} with MSE: {}".format(round(growth_rates[np.argmin(mses)], 3), np.min(mses)))
    add_grs = np.append(add_grs, round(growth_rates[np.argmin(mses)], 3))

Add the 100% experiment growth rates to the list of additive growth rates

In [ ]:
add_grs = np.append(0.135, add_grs)
add_grs = np.append(add_grs, 0.107)

## Plot

Plot the growth rates versus the admixture ratio for both the additive model and the growth rates determined directly from the data. Save if save file is specified.

In [ ]:
plt.figure()
plt.plot([1, 0.8, 0.5, 0.2, 0], admix_grs, label="Fit to data", color="black", linestyle="solid", marker="o", markersize=8)
plt.plot([1, 0.8, 0.5, 0.2, 0], add_grs, label="Additive", color="black", linestyle="dashed", marker="^", markersize=9)
plt.ylim([0.099, .141])
plt.grid(True)
plt.legend()
plt.gca().invert_xaxis()
plt.xlabel("Ratio of C1")
plt.ylabel("Estimated growth rate")
if fig_save_file:
    plt.savefig(fig_save_file)
plt.show()

# Estimate growth rates for B6 mice (Supplementary Table 1)

In [ ]:
result_b6_path = "results/growth_b6/"

In [ ]:
results_b6 = []
for fname in [f for f in os.listdir(result_b6_path) if ".csv" in f]:
    results_b6 += [pd.read_csv("{}/{}".format(result_b6_path, fname))]
results_b6 = pd.concat(results_b6)
results_b6

In [ ]:
avgs_b6 = results_b6.groupby(["group", "g"])["error"].mean().reset_index()
min_b6 = avgs_b6.groupby("group").apply(lambda x: x[x["error"] == x["error"].min()])
min_b6